In [ ]:
import sys
import os

#####################
# Import of utils.py functions
#####################
# Required to get utils.py and access its functions
notebook_dir = os.path.abspath("")
parent_dir = os.path.abspath(os.path.join(notebook_dir, '..'))
sys.path.append(parent_dir)
sys.path.append('.')
from utils import loadFSL, FSLeyesServer, mkdir_no_exist, interactive_MCQ

####################
# DIPY_HOME should be set prior to import of dipy to make sure all downloads point to the right folder
####################
os.environ["DIPY_HOME"] = "/home/jovyan/Data"


#############################
# Loading fsl and freesurfer within Neurodesk
# You can find the list of available other modules by clicking on the "Softwares" tab on the left
#############################
import module as lmod
await lmod.purge(force=True)
await lmod.load('fsl/6.0.7.18')
await lmod.load('freesurfer/7.4.1')
await lmod.list()

####################
# Setup FSL path
####################
loadFSL()

###################
# Load all relevant libraries for the lab
##################
import fsl.wrappers
from fsl.wrappers import fslmaths

import mne_nirs
import nilearn
from nilearn.datasets import fetch_development_fmri

import mne
import mne_nirs

import xml.etree.ElementTree as ET
import os.path as op
import nibabel as nib
import glob

!pip install antspyx
import ants

import openneuro
from mne.datasets import sample
from mne_bids import BIDSPath, read_raw_bids, print_dir_tree, make_report

# Useful imports to define the direct download function below
import requests
import urllib.request
from tqdm import tqdm


# FSL function wrappers which we will call from python directly
from fsl.wrappers import fast, bet
from fsl.wrappers.misc import fslroi
from fsl.wrappers import flirt

# General purpose imports to handle paths, files etc
import glob
import pandas as pd
import numpy as np
import json
fMRI_data_root = op.join(notebook_dir, 'fMRI', 'tfMRI_LANGUAGE_LR')
fMRI_data_filepath = op.join(fMRI_data_root, 'tfMRI_LANGUAGE_LR.nii.gz')

In [ ]:
%gui wx

In [ ]:
################
# Start FSLeyes (very neat tool to visualize MRI data of all sorts) within Python
################
fsleyesDisplay = FSLeyesServer()
fsleyesDisplay.show()

In [ ]:
### Removing problematic volumes
# 1) remove first volumes, we need to see when the signal settles
import matplotlib.pyplot as plt
import nibabel as nib

fMRI_data_full = nib.load(op.join(fMRI_data_filepath)).get_fdata()
total_nb_volumes = fMRI_data_full.shape[3]
print(total_nb_volumes)
plt.plot(fMRI_data_full.mean(axis=(0,1,2)))
plt.xlabel('Time (volume)')
plt.ylabel('Mean voxel intensity')

In [ ]:
## We see the first volume is a big drop, but as we did in lab 3, we will remove the first 10 to be sure.
file_to_trim = glob.glob(fMRI_data_filepath)[0]
derivatives_folder = op.join(notebook_dir, 'derivatives')
output_target = op.join(derivatives_folder, 'tfMRI_LANGUAGE_LR_settled.nii.gz')

In [ ]:
# For this, knowing that there are originally 184 volumes and that you want to throw away the first 10, please fill in
# the following variables
start_vol = 9 # Where should we start? (First volume is 0, not 1 !)
number_of_volumes = total_nb_volumes - start_vol

# Function creating new nifti with trimmed number of volumes
fslroi(file_to_trim, output_target, str(start_vol), str(number_of_volumes))
print(f"You kept {nib.load(output_target).shape[-1]} volumes.")

In [ ]:
from fsl.wrappers import mcflirt

# Input: your fMRI file
fMRI_settled_data_filepath = output_target

# Output: a derivatives folder next to the data
preproc_root = op.join(notebook_dir, 'derivatives')
os.makedirs(preproc_root, exist_ok=True)

path_moco_data = op.join(preproc_root, 'tfMRI_LANGUAGE_LR_moco')

In [ ]:
mcflirt(infile=fMRI_settled_data_filepath, o=path_moco_data, plots=True, report=True, dof=6, mats=True)

In [ ]:
print_dir_tree(preproc_root, max_depth=5)

In [ ]:
fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(fMRI_data_filepath)
fsleyesDisplay.load(path_moco_data)

In [ ]:
def load_mot_params_fsl_6_dof(path):
    return pd.read_csv(path, sep='  ', header=None, 
            engine='python', names=['Rotation x', 'Rotation y', 'Rotation z','Translation x', 'Translation y', 'Translation z'])

mot_params = load_mot_params_fsl_6_dof(op.join(preproc_root, 'tfMRI_LANGUAGE_LR_moco.par'))
mot_params

In [ ]:
def compute_FD_power(mot_params):
    framewise_diff = mot_params.diff().iloc[1:]

    rot_params = framewise_diff[['Rotation x', 'Rotation y', 'Rotation z']]
    # Estimating displacement on a 50mm radius sphere
    # To know this one, we can remember the definition of the radian!
    # Indeed, let the radian be theta, the arc length be s and the radius be r.
    # Then theta = s / r
    # We want to determine here s, for a sphere of 50mm radius and knowing theta. Easy enough!
    
    # Another way to think about it is through the line integral along the circle.
    # Integrating from 0 to theta with radius 50 will give you, unsurprisingly, r0 theta.
    converted_rots = rot_params*50
    trans_params = framewise_diff[['Translation x', 'Translation y', 'Translation z']]
    fd = converted_rots.abs().sum(axis=1) + trans_params.abs().sum(axis=1)
    return fd

fd = compute_FD_power(mot_params).to_numpy()

In [ ]:
threshold = np.quantile(fd,0.75) + 1.5*(np.quantile(fd,0.75) - np.quantile(fd,0.25))

In [ ]:
#%matplotlib inline
plt.plot(list(range(1, fd.size+1)), fd)
plt.xlabel('Volume')
plt.ylabel('FD displacement (mm)')
plt.hlines(threshold, 0, 370,colors='black', linestyles='dashed', label='FD threshold')
plt.legend()
plt.show()

In [ ]:
print(f"Indices of frames above threshold: {(np.where(fd > threshold)[0] + 1).tolist()}")

In [ ]:
#The patient did not have anz significant movement. The FWD is alwazs below the threshold

In [ ]:
## NOW WE DO Registration from Functional to anatomical

In [ ]:
# We need skull stripped T1 & perform tissue segmentation for registration
import subprocess
whole_t1 = os.path.join(notebook_dir, 'T1', 'T1w.nii.gz')
skull_stripped_t1 = op.join(derivatives_folder, 'T1w_brain')
white_matter_segmentation = op.join(derivatives_folder, "T1w_fast_pve_2") # two is white matter
## use motion corrected as source for registration
epi_source = path_moco_data
output_midvol_2_anat = op.join(derivatives_folder, "tfMRI_LANGUAGE_LR_moco_midvol_2_anat")

In [ ]:
#use the middle moco volume as reference, use fslroi to extract it
ref_vol_name = op.join(derivatives_folder, "tfMRI_LANGUAGE_LR_moco_midvol")

In [ ]:
fslroi(path_moco_data, ref_vol_name, str(number_of_volumes // 2), '1')
# run the registration
subprocess.run(['epi_reg','--epi={}'.format(ref_vol_name), '--t1={}'.format(whole_t1), '--t1brain={}'.format(skull_stripped_t1), 
                '--out={}'.format(output_midvol_2_anat),
               '--wmseg={}'.format(white_matter_segmentation)])
# Now we have registered fMRI to the individual's structural, 
# but we still need to bring fMRI to MNI, to we apply the MNI matrix now

In [ ]:
## visualize alignement before MNI matrix is applied to functional
fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(skull_stripped_t1)
fsleyesDisplay.load(output_midvol_2_anat)

In [ ]:
# apply MNI matrix
func_2_anat_mat = output_midvol_2_anat + '.mat'                         # epi_reg: func -> T1
anat_2_mni_mat  = op.join(derivatives_folder, 'T1w_brain_2_mni.mat')   # flirt:   T1   -> MNI
mni_template    = op.expandvars(op.join('$FSLDIR', 'data', 'standard', 'MNI152_T1_1mm_brain'))
func_2_mni_mat  = op.join(derivatives_folder, 'tfMRI_LANGUAGE_LR_moco_2_mni.mat')
midvol_mni      = op.join(derivatives_folder, 'tfMRI_LANGUAGE_LR_moco_midvol_2_mni')

subprocess.run(['convert_xfm', '-omat', func_2_mni_mat,
                '-concat', anat_2_mni_mat, func_2_anat_mat])
subprocess.run(['flirt', '-in', ref_vol_name, '-ref', mni_template,
                '-applyxfm', '-init', func_2_mni_mat, '-out', midvol_mni])

In [ ]:
#fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(op.join(derivatives_folder, 'T1w_brain_2_mni'))   # T1w_brain_2_mni        # T1w_brain_2_mni
fsleyesDisplay.load(midvol_mni)

In [ ]:
## now we do smoothing on the MNI aligned, but how to choose fwhm ?

In [ ]:
fwhm = 6
cmd = 'fslmaths {} -s {} {}_smoothed-{}mm'.format(output_func_2_anat, fwhm/2.3548, output_func_2_anat, fwhm)
smoothed_fmri_filename = '{}_smoothed-{}mm'.format(output_func_2_anat, fwhm)
subprocess.run(['fslmaths',output_func_2_anat, '-s', str(fwhm/2.3548), smoothed_fmri_filename])

In [ ]:
## visualize smoothed brain fmri

In [ ]:
fsleyesDisplay.load(smoothed_fmri_filename)